In [1]:
from datasets import load_dataset

dataset = load_dataset("mosaicml/instruct-v3")
dataset

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/56167 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6807 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response', 'source'],
        num_rows: 56167
    })
    test: Dataset({
        features: ['prompt', 'response', 'source'],
        num_rows: 6807
    })
})

In [2]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

train_df

,prompt,response,source
0,Below is an instruction that describes a task....,On Wednesday she covered half of the distance ...,cot_gsm8k
1,Below is an instruction that describes a task....,"There are more than 12,000 species of grass. T...",dolly_hhrlhf
2,Below is an instruction that describes a task....,There are a total of 10 x 30 = 300 pieces of o...,cot_gsm8k
3,Below is an instruction that describes a task....,Let $w$ be the complex number corresponding to...,competition_math
4,Below is an instruction that describes a task....,The best way to cook food is over a fire. You’...,dolly_hhrlhf
...,...,...,...
56162,Below is an instruction that describes a task....,"Pasta = casarecce, gnocchi, tonnarelli, orecch...",dolly_hhrlhf
56163,Below is an instruction that describes a task....,Applying for a credit card can lower your cred...,dolly_hhrlhf
56164,Below is an instruction that describes a task....,"To become a cheerleader, you’ll need to meet t...",dolly_hhrlhf
56165,Below is an instruction that describes a task....,Three times,dolly_hhrlhf


In [6]:
# reformat instr
# Below is an instruction that describes a task. Write a response that appropriately completes the request. ### Instruction How do you know if you have gout? ### Response	
# to:
# How do you know if you have gout?

def format_prompt(example, instr_format="### Instruction", resp_format="### Response"):
    text = example["prompt"]
    _, instruction = text.split(instr_format)
    instruction, response = instruction.split(resp_format)
    instruction = instruction.strip()
    return instruction

train_df["instruction"] = train_df.apply(format_prompt, axis=1)
test_df["instruction"] = test_df.apply(format_prompt, axis=1)

In [9]:
# use the mistral tokenizer
from transformers import AutoTokenizer
base_model_id = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    padding_side="left",
    add_eos_token=True,
    add_bos_token=True,
)
tokenizer.pad_token = tokenizer.eos_token

In [12]:
train_df.iloc[0]

prompt         Below is an instruction that describes a task....
response       On Wednesday she covered half of the distance ...
source                                                 cot_gsm8k
instruction    Question: Nancy and Rose are making bracelets,...
Name: 0, dtype: object

In [15]:
def make_prompt(inst, out):
    return f"""<s>[INST] {inst} [/INST] \\n {out} </s>"""

def tknz(example, max_length=768):
    inst = example["instruction"]
    out = example["response"]
    inst = inst if inst is not None else ""
    out = out if out is not None else ""
    # return tokenizer(make_prompt(inst, inp, out), truncation=True, max_length=max_length, padding="max_length")
    return tokenizer(make_prompt(inst, out))

# demo the prompt:
print(make_prompt("Jeg vil at du skal sortere følgende: 6 3 1 2", out="1 2 3 6"))

tokenized_train = train_df.apply(tknz, axis=1)
tokenized_test = test_df.apply(tknz, axis=1)

<s>[INST] Jeg vil at du skal sortere følgende: 6 3 1 2 [/INST] \n 1 2 3 6 </s>


In [20]:
# add the length of the input_ids from the tokenizer to the original dfs

train_df["tokenized_len"] = tokenized_train.apply(lambda x: len(x["input_ids"]))
test_df["tokenized_len"] = tokenized_test.apply(lambda x: len(x["input_ids"]))

In [23]:
print(train_df.shape, test_df.shape)
train_df = train_df[train_df["tokenized_len"] <= 1024]
test_df = test_df[test_df["tokenized_len"] <= 1024]
print(train_df.shape, test_df.shape)

(56167, 5) (6807, 5)
(45767, 5) (5933, 5)


In [25]:
# keep only instruction, response and source
train_df = train_df[["instruction", "response", "source"]]
test_df = test_df[["instruction", "response", "source"]]
train_df

,instruction,response,source
1,What are different types of grass?,"There are more than 12,000 species of grass. T...",dolly_hhrlhf
3,"The points $P,$ $Q,$ and $R$ are represented b...",Let $w$ be the complex number corresponding to...,competition_math
4,How can I cook food while camping?,The best way to cook food is over a fire. You’...,dolly_hhrlhf
5,What are some fun scenarios my kids can play w...,Some fun scenarios for your kids to play with ...,dolly_hhrlhf
6,How many titles have Liverpool won?\nDomestica...,"Liverpool has won 19 League titles, 8 FA cups,...",dolly_hhrlhf
...,...,...,...
56162,Classify each of the following as either pasta...,"Pasta = casarecce, gnocchi, tonnarelli, orecch...",dolly_hhrlhf
56163,Why does applying for a credit card make your ...,Applying for a credit card can lower your cred...,dolly_hhrlhf
56164,how do i become an nfl cheerleader?,"To become a cheerleader, you’ll need to meet t...",dolly_hhrlhf
56165,How many times did Barton switch parties?\nBar...,Three times,dolly_hhrlhf


In [31]:
print(train_df.shape, test_df.shape)
# remove rows with instruction or responses longer than 512 words
train_df = train_df[train_df["instruction"].apply(lambda x: len(x.split()) <= 512)]
train_df = train_df[train_df["response"].apply(lambda x: len(x.split()) <= 512)]

test_df = test_df[test_df["instruction"].apply(lambda x: len(x.split()) <= 512)]
test_df = test_df[test_df["response"].apply(lambda x: len(x.split()) <= 512)]
print(train_df.shape, test_df.shape)

(45767, 3) (5933, 3)
(44457, 3) (5829, 3)


In [40]:
from transformers import pipeline

pipe = pipeline("translation", model="facebook/nllb-200-distilled-1.3B", device="cuda:0")

In [ ]:
train_instr = train_df["instruction"].tolist()
train_res = train_df["response"].tolist()

test_instr = test_df["instruction"].tolist()
test_res = test_df["response"].tolist()

In [41]:
# subset of train_df n=10
sample_df = train_df.sample(10)
sample_df

,instruction,response,source
18826,What's a fun board game to play with the whole...,You've come to the right place! I can certainl...,dolly_hhrlhf
23776,Is Dr. Seus a good book to read to children?,"I can’t speak to that personally, but Dr. Seus...",dolly_hhrlhf
26238,Can you give me some tips on how I can prepare...,"Sure. First, you’ll probably want to look at ...",dolly_hhrlhf
54824,Does diabetes run in the family?,"Yes, there’s a strong genetic basis for diabet...",dolly_hhrlhf
33941,Have the Detroit Redwings won any Stanley Cups?,Not since 1997. They are currently ranked fift...,dolly_hhrlhf
52615,How do solar panels create energy?,Solar panels convert the radiant energy from s...,dolly_hhrlhf
8099,Is vaping better for you then cigarettes?,Vaping can help you quit smoking if you use it...,dolly_hhrlhf
6235,Identify which instrument is string or percuss...,"Baglama is string, Yangqin is percussion.",dolly_hhrlhf
45249,Who are some notable guests on Between Two Ferns?,"Between Two Ferns, a show where Zach Galifiana...",dolly_hhrlhf
30431,What can I cook with if I don't have garlic?,"Most soups, stews and sauces can be made with ...",dolly_hhrlhf


In [45]:
from tqdm import tqdm

LANGS = {
    "EN": "eng_Latn",
    "NOB": "nob_Latn"
}

def translate(pipe, datamapper, src_lang=LANGS["EN"], tgt_lang=LANGS["NOB"], batch_size=64):
    return pipe(datamapper, src_lang=src_lang, tgt_lang=tgt_lang, batch_size=batch_size)

def translate_df(_df):
    def datamapper(key):
        for _, d in _df.iterrows():
            yield d[key]

    for sent in ["instruction", "response"]:
        translator = translate(pipe=pipe, datamapper=datamapper(key=sent))
        translated = []
        for x in tqdm(translator):
            translated.append(x[0]["translation_text"])
        _df[sent] = translated
    
    return _df

translated = {
    "train": translate_df(train_df),
    "test": translate_df(test_df)
}

# save the translated data
for k, v in translated.items():
    v.to_csv(f"{k}.csv", index=False)

0it [00:00, ?it/s]Your input_length: 701 is bigger than 0.9 * max_length: 200. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
1it [00:12, 12.16s/it]Your input_length: 743 is bigger than 0.9 * max_length: 200. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
65it [00:24,  3.09it/s]Your input_length: 632 is bigger than 0.9 * max_length: 200. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
129it [00:35,  4.23it/s]Your input_length: 713 is bigger than 0.9 * max_length: 200. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
192it [00:42,  4.54it/s]


KeyboardInterrupt: 

In [37]:
# sample of 100 texts
train_instr_sample = train_instr[:10]
translated = translate(pipe, train_instr_sample)
translated

Your input_length: 744 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


[{'translation_text': 'Hva er de forskjellige typer gress?'},
 {'translation_text': 'Punktene $P, $ $Q, $ og $R$ er representert av de komplekse tallene $z, $ $(1 + i) z, $ og $2 \\overline{z}, $ henholdsvis, hvor $                                                                                                                                                                                                                                      '},
 {'translation_text': 'Hvordan kan jeg lage mat mens jeg camper?'},
 {'translation_text': 'Hva er noen morsomme scenarier mine barn kan leke med sine Barbie-dunker?'},
 {'translation_text': 'Hvor mange titler har Liverpool vunnet? På hjemmebane har klubben vunnet 19 liga titler, åtte FA Cups, en rekord ni League Cups og 16 FA Community Shields.'},
 {'translation_text': 'Er det vanskelig å bli øyelege?'},
 {'translation_text': 'Den skrå høyde av en kjegle er 13 cm, og høyden fra toppen til midten av basen er 12 cm. Hva er antall kubikkcentimeter 

In [ ]:
from transformers import pipeline
pipe = pipeline("translation", model="Helsinki-NLP/opus-mt-en-gmq")

def translate